# 1. Convert public seizure datasets to EEG-BIDS

Bids2SeizureML reads datasets stored in EEG-BIDS. This notebook converts the original
datasets with [epilepsy2bids](https://github.com/esl-epfl/epilepsy2bids) (version 0.0.7 was
used here: `pip install epilepsy2bids==0.0.7`).

The examples use the **Siena Scalp EEG Database**: it is openly available, small (14 patients,
41 recordings) and converts in a few minutes. CHB-MIT and TUSZ are converted in the same way.

| Dataset | Download | Source folder to give below |
| :-- | :-- | :-- |
| Siena | https://physionet.org/content/siena-scalp-eeg/1.0.0/ | the folder containing `PN00`, `PN01`, … and `subject_info.csv` |
| CHB-MIT | https://physionet.org/content/chbmit/1.0.0/ | the folder containing `chb01`, `chb02`, … and `SUBJECT-INFO` |
| TUSZ | https://isip.piconepress.com/projects/nedc/html/tuh_eeg/ (free registration) | the `edf` folder containing `train`, `dev` and `eval` |

What the converter does:

- **Siena and TUSZ:** keeps the 19 electrodes of the 10–20 system, re-references them to the
  common average and resamples to 256 Hz. A missing electrode is replaced by zeros and reported
  with a `Missing electrode … replaced by zeros` message.
- **CHB-MIT:** keeps the 18 channels of the longitudinal bipolar (double-banana) montage at 256 Hz.
- Each recording gets a JSON sidecar (metadata) and an `events.tsv` file (seizure annotations).

## Settings

In [ ]:
from pathlib import Path

# Folders of the original downloads (edit these).
SOURCE_DIRS = {
    "siena": Path("/path/to/siena-scalp-eeg/1.0.0"),
    "chbmit": Path("/path/to/chbmit/1.0.0"),
    "tusz": Path("/path/to/tusz/edf"),
}

# Each converted dataset is written to BIDS_ROOT/<name>.
BIDS_ROOT = Path("../data")

# Datasets to convert: any of "siena", "chbmit" and "tusz".
SELECTED = ["siena"]

## Convert

The converter prints one line per patient folder. 

In [ ]:
import warnings

from epilepsy2bids.bids.chbmit.convert2bids import convert as convert_chbmit
from epilepsy2bids.bids.siena.convert2bids import convert as convert_siena
from epilepsy2bids.bids.tuh.convert2bids import convert as convert_tusz

CONVERTERS = {"siena": convert_siena, "chbmit": convert_chbmit, "tusz": convert_tusz}

for name in SELECTED:
    source, target = SOURCE_DIRS[name], BIDS_ROOT / name
    if not source.is_dir():
        raise FileNotFoundError(f"{name}: source folder not found: {source}")
    target.mkdir(parents=True, exist_ok=True)
    print(f"Converting {name} into {target} ...")
    with warnings.catch_warnings():
        # pyedflib warns when a signal range is rounded for the EDF header; this is harmless.
        warnings.simplefilter("ignore", UserWarning)
        CONVERTERS[name](source, target)
    print(f"Finished {name}")

## Check the result

In [ ]:
import json

import pandas as pd
from IPython.display import display

for name in SELECTED:
    recordings = sorted((BIDS_ROOT / name).glob("sub-*/ses-*/eeg/*_eeg.edf"))
    subjects = {path.parts[-4] for path in recordings}
    print(f"{name}: {len(subjects)} subjects, {len(recordings)} recordings")

# Metadata and annotations of the first recording of the last dataset converted.
example = recordings[0]
print(f"\n{example.name}")
print(json.dumps(json.loads(example.with_suffix(".json").read_text()), indent=2))
display(pd.read_csv(example.with_name(example.name.replace("_eeg.edf", "_events.tsv")), sep="\t"))